# Full-specification estimates of the problem-set models
Same models as `pset_quick_estimates.ipynb`, with a tight contraction tolerance, many starting points and more optimizer iterations. Starts run in parallel (start Julia with `-t auto`). Estimates are saved in `models_full/`; the best of the starts is kept and starts that error are logged in `models_full/*_failed_starts.jdl2`.

In [1]:
using MAT, DataFrames, Statistics, LinearAlgebra, JLD2
include("blp_optimize/blp_new.jl")
using .blp
Threads.nthreads()

1

## Settings

In [2]:
const TOLER = 1e-10     # contraction tolerance (quick: 1e-4)
const MAX_ITER = 10_000 # contraction iterations (quick: 1000)
const NS = 16           # starting points (quick: 1)
const ITERATIONS = 100  # optimizer iterations per start (quick: 15)
const TIME_LIMIT = 600.0 # seconds per start; a start that hits it returns its best point so far
const OUTDIR = "models_full"
mkpath(OUTDIR)

beta = [5, 1, 1]; alpha_bar = -1; gamma = [2, 1, 1]; sigma = 1.0

1.0

## Data

In [3]:
simdir = "../build/input/public/simulation"
d1 = matread(joinpath(simdir, "10markets3products.mat"))
d2 = matread(joinpath(simdir, "100markets3products.mat"))

function process_data(d)
    J, T = size(d["P_opt"])
    p = reshape(d["P_opt"], (T*J, 1)); shares = reshape(d["shares"], (T*J, 1))
    data = blp.BLPData(T, J, d["x1"], p, shares, log.(d["alphas"] .- 1))
    model = blp.BLPDemand(BLPData = data)
    model.xi_real = d["xi_all"]
    model.cost = gamma[1] .+ (gamma[2] * d["w"] + gamma[3] * d["Z"] + d["eta"])
    model.markup .= ((model.p - model.cost) ./ model.p)
    model.profit = (model.p - model.cost) .* model.shares
    return model
end

# Estimates a model with the quick settings and saves it
function estimate!(model, name)
    println("Estimating $name ..."); flush(stdout)
    failures = []   # starts that errored: (sigma0, error); the search continues with the other starts
    kw = (toler = TOLER, max_iter = MAX_ITER, Ns = NS, iterations = ITERATIONS, time_limit = TIME_LIMIT, failures = failures)
    t = @elapsed if model isa blp.BLPDemand
        blp.estimate_blp!(model; kw...)
    else
        blp.estimate_blp_joint!(model; kw...)
    end
    jldsave(joinpath(OUTDIR, "$name.jdl2"); model = model)
    jldsave(joinpath(OUTDIR, "$(name)_failed_starts.jdl2"); failures = failures)
    println("$name: $(length(failures)) of $NS starts failed")
    println("$name: $(round(t, digits = 1))s, params = ", round.(vec(model.params), digits = 4))
    return model
end

timings = Dict{String, Float64}()
set_Z!(m, Z) = (m.Z = Z; m.W = inv(Z' * Z); m)

set_Z! (generic function with 1 method)

In [4]:
model = process_data(d2)
model.beta .= vcat(beta, [alpha_bar]); model.sigma = sigma
blp.simulate_BLP!(model, delta = model.X_all * model.beta + model.xi_real, sigma = sigma, all = true)

price_bar = blp.compute_hausman(model)
X_rivals = blp.compute_blp(model)
set_Z!(model, hcat(model.X, X_rivals))
timings["model_nh"] = @elapsed model = estimate!(model, "model_nh")

Estimating model_nh ...
Reached max number of iterations
  start sigma0 = 0.594 -> sigma = 1.0543, obj = 0.0052321470948879355, 36.3s
  start sigma0 = 0.908 -> sigma = 1.0543, obj = 0.005232147094887021, 26.3s
  start sigma0 = 0.859 -> sigma = 1.0543, obj = 0.005232147094886812, 26.1s
  start sigma0 = 1.426 -> sigma = 1.0543, obj = 0.0052321470948874455, 32.0s
Reached max number of iterations
  start sigma0 = 0.182 -> sigma = 1.0543, obj = 0.005232147094887644, 30.5s
  start sigma0 = 0.876 -> sigma = 1.0543, obj = 0.005232147094889682, 26.3s
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
  start sigma0 = 0.254 -> sigma = 1.0543, obj = 0.005232147094890379, 48.8s
  start sigma0 = 1.064 -> sigma = 1.0543, obj = 0.005232147094885582, 17.7s
Reached max number of iterations
Reached max number of iterations
  start sigma0 = 0.228 -> sigma = 1.0543, obj = 0.0052321470948852675, 42.6s
  start sigma0 = 1.168 -> sigma = 1.0543, obj = 0.00523214

┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


model_nh: 0 of 16 starts failed
model_nh: 506.0s, params = [5.5894, 1.0998, 1.0815, -1.1685, 1.0543]


510.485012292

In [5]:
model_ls = process_data(d1)
p_bar_ls = blp.compute_hausman(model_ls); X_rivals_ls = blp.compute_blp(model_ls)
set_Z!(model_ls, hcat(model_ls.X, X_rivals_ls))
timings["model_ls_nh"] = @elapsed model_ls = estimate!(model_ls, "model_ls_nh")

Estimating model_ls_nh ...
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max number of iterations
Reached max numb

┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


649.784378625

In [6]:
model_inc = deepcopy(model)
set_Z!(model_inc, hcat(model_inc.X_all, X_rivals))
timings["model_inc_nh"] = @elapsed model_inc = estimate!(model_inc, "model_inc_nh")

Estimating model_inc_nh ...
Reached max number of iterations
Reached max number of iterations
  start sigma0 = 0.594 -> sigma = 0.9895, obj = 0.005831448962240217, 52.8s
Reached max number of iterations
  start sigma0 = 0.908 -> sigma = 0.9895, obj = 0.00583144896223991, 39.3s
Reached max number of iterations
  start sigma0 = 0.859 -> sigma = 0.9895, obj = 0.005831448962240326, 37.6s
  start sigma0 = 1.426 -> sigma = 0.9895, obj = 0.005831448962240355, 50.6s
Reached max number of iterations
  start sigma0 = 0.182 -> sigma = 0.9895, obj = 0.005831448962239455, 42.7s
Reached max number of iterations
  start sigma0 = 0.876 -> sigma = 0.9895, obj = 0.005831448962240452, 37.4s
Reached max number of iterations
  start sigma0 = 0.254 -> sigma = 0.9895, obj = 0.005831448962237161, 43.1s
  start sigma0 = 1.064 -> sigma = 0.9895, obj = 0.005831448962239596, 32.6s
Reached max number of iterations
  start sigma0 = 0.228 -> sigma = 0.9895, obj = 0.005831448962237837, 37.1s
  start sigma0 = 1.168 ->

┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447
┌ Warning: Attempting to store Optim.var"#Fminbox##0#Fminbox##1".
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


model_inc_nh: 0 of 16 starts failed
model_inc_nh: 671.6s, params = [4.7753, 0.9218, 1.0216, -0.9039, 0.9895]


┌ Warning: Attempting to store Optim.var"#optimize##0#optimize##1"{Vector{Float64}, Vector{Int64}, Optim.Fminbox{Optim.LBFGS{Nothing, LineSearches.InitialStatic{Float64}, LineSearches.HagerZhang{Float64, Base.RefValue{Bool}}, Returns{Nothing}}, Float64, Optim.var"#Fminbox##0#Fminbox##1"}}.
│ JLD2 only stores functions by name.
│  This may not be useful for anonymous functions.
└ @ JLD2 ~/.julia/packages/JLD2/rm5a0/src/data/writing_datatypes.jl:447


671.589011

In [ ]:
error 
mistake

## Baseline: T = 100, J = 3 (Hausman price instrument + rival characteristics)

In [ ]:
model = process_data(d2)
model.beta .= vcat(beta, [alpha_bar]); model.sigma = sigma
blp.simulate_BLP!(model, delta = model.X_all * model.beta + model.xi_real, sigma = sigma, all = true)

price_bar = blp.compute_hausman(model)
X_rivals = blp.compute_blp(model)
set_Z!(model, hcat(model.X, price_bar, X_rivals))
timings["model"] = @elapsed model = estimate!(model, "model")

## T = 10, J = 3

In [ ]:
model_ls = process_data(d1)
p_bar_ls = blp.compute_hausman(model_ls); X_rivals_ls = blp.compute_blp(model_ls)
set_Z!(model_ls, hcat(model_ls.X, p_bar_ls, X_rivals_ls))
timings["model_ls"] = @elapsed model_ls = estimate!(model_ls, "model_ls")

## Price as its own instrument

In [ ]:
model_inc = deepcopy(model)
set_Z!(model_inc, hcat(model_inc.X_all, price_bar, X_rivals))
timings["model_inc"] = @elapsed model_inc = estimate!(model_inc, "model_inc")

## Adding the cost shifter w as a demand instrument

In [ ]:
model_ss = deepcopy(model)
set_Z!(model_ss, hcat(model_ss.X, price_bar, X_rivals, d2["w"]))
timings["model_ss"] = @elapsed model_ss = estimate!(model_ss, "model_ss")

model_ls_ss = deepcopy(model_ls)
set_Z!(model_ls_ss, hcat(model_ls_ss.X, p_bar_ls, X_rivals_ls, d1["w"]))
timings["model_ls_ss"] = @elapsed model_ls_ss = estimate!(model_ls_ss, "model_ls_ss")

## Joint demand + supply under three conduct assumptions

In [ ]:
X_supply = hcat(ones(model.T * model.J, 1), d2["Z"], d2["w"])
Z_supply = hcat(model.X, X_rivals, X_supply[:, 2:end])
model_supply = blp.BLPSupply(X = X_supply, Z = Z_supply, W = inv(Z_supply' * Z_supply))

systems = Dict{String, blp.BLP}()
for (name, Omega) in (("bert", Matrix{Float64}(I, model.J, model.J)), ("coll", ones(model.J, model.J)), ("comp", zeros(model.J, model.J)))
    dem = deepcopy(model_ss)
    dem.Omega = Omega
    sys = blp.BLP(BLPDemand = dem, BLPSupply = deepcopy(model_supply))
    timings["system_$name"] = @elapsed estimate!(sys, "model_system_$name")
    systems[name] = sys
end

## Summary

In [ ]:
summary = DataFrame(
    model = ["model", "model_ls", "model_inc", "model_ss", "model_ls_ss"],
    sigma = [m.params[end] for m in (model, model_ls, model_inc, model_ss, model_ls_ss)],
    price_coef = [m.beta[end] for m in (model, model_ls, model_inc, model_ss, model_ls_ss)],
    seconds = [round(timings[n], digits = 1) for n in ("model", "model_ls", "model_inc", "model_ss", "model_ls_ss")])

In [ ]:
summary_sys = DataFrame(
    model = ["bert", "coll", "comp"],
    sigma = [systems[n].params[end] for n in ("bert", "coll", "comp")],
    price_coef = [systems[n].BLPDemand.beta[end] for n in ("bert", "coll", "comp")],
    seconds = [round(timings["system_$n"], digits = 1) for n in ("bert", "coll", "comp")])
summary, summary_sys